# CasCrop: Temporal Cascade Experiments
**All experiments for Nature Communications submission.**

4 temporal models × 3 seeds × 50 epochs on 638K monthly samples.
~1-2 hours on T4 GPU.

In [ ]:
#@title Configuration
QUICK_TEST = True  #@param {type:"boolean"}
SEEDS = [42, 123, 456] if QUICK_TEST else [42, 123, 456, 789, 1024]
EPOCHS = 30 if QUICK_TEST else 100
PATIENCE = 10 if QUICK_TEST else 20
BATCH_SIZE = 2048 if QUICK_TEST else 1024
print(f"{'QUICK' if QUICK_TEST else 'FULL'}: {len(SEEDS)} seeds, {EPOCHS} epochs")

In [ ]:
#@title Setup
import torch, os, sys, json, time, shutil, subprocess
import numpy as np, pandas as pd
from pathlib import Path

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f}GB)')
    if gpu_mem < 8: BATCH_SIZE = min(BATCH_SIZE, 512)
else:
    print('NO GPU - Runtime > Change runtime type > T4')

if not os.path.exists('CasCrop'):
    !git clone https://github.com/keshavkrishnan08/CasCrop.git
if os.path.basename(os.getcwd()) != 'CasCrop':
    os.chdir('CasCrop')
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn tqdm pyyaml 2>&1 | tail -1
sys.path.insert(0, 'src')
for d in ['checkpoints','results','paper/figures','paper/tables']:
    os.makedirs(d, exist_ok=True)

# Drive mount (optional)
SAVE_TO_DRIVE = False; DRIVE_PATH = ''
try:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/CasCrop_Results'
    os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
    for f in Path(f'{DRIVE_PATH}/checkpoints').glob('*.pt'):
        if not Path(f'checkpoints/{f.name}').exists(): shutil.copy2(f, f'checkpoints/{f.name}')
    SAVE_TO_DRIVE = True; print(f'Drive: {DRIVE_PATH}')
except: print('No Drive')

def backup():
    if not SAVE_TO_DRIVE: return
    for d in ['results','checkpoints','paper/figures','paper/tables']:
        if not os.path.exists(d): continue
        dst = f'{DRIVE_PATH}/{d}'; os.makedirs(dst, exist_ok=True)
        for f in Path(d).glob('*'):
            if f.is_file(): shutil.copy2(f, f'{dst}/{f.name}')
print('OK')

In [ ]:
#@title Load Monthly Data (68MB auto-download)
if Path('data/processed/features_monthly.parquet').exists() and Path('data/graphs/combined_graph.npz').exists():
    print('Data present.')
else:
    !wget -q --show-progress -O monthly.tar.gz https://github.com/keshavkrishnan08/CasCrop/releases/download/v0.1-data/cascrop_monthly.tar.gz
    !tar xzf monthly.tar.gz && rm monthly.tar.gz

# Verify
assert Path('data/processed/features_monthly.parquet').exists(), 'Monthly features missing!'
assert Path('data/processed/labels_monthly.parquet').exists(), 'Monthly labels missing!'
assert Path('data/processed/splits_monthly.json').exists(), 'Monthly splits missing!'
assert Path('data/graphs/combined_graph.npz').exists(), 'Graph missing!'

f = pd.read_parquet('data/processed/features_monthly.parquet')
l = pd.read_parquet('data/processed/labels_monthly.parquet')
with open('data/processed/feature_groups_monthly.json') as fj: groups = json.load(fj)
print(f'{len(f):,} samples | {f["fips"].nunique()} counties | {l["waste"].mean():.1%} waste')
print(f'bio={len(groups["biophysical"])}, econ={len(groups["economic"])}, hist={len(groups["historical"])}')

In [ ]:
#@title Smoke Test (import + forward + backward on GPU)
import importlib
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
_reg = {
    'temporal_local': ('models.baselines.temporal_local', 'TemporalLocalModel'),
    'temporal_gat': ('models.baselines.temporal_gat', 'TemporalGATModel'),
    'temporal_symmetric': ('models.baselines.temporal_symmetric', 'TemporalSymmetricModel'),
    'temporal_cascrop': ('models.temporal_cascrop', 'TemporalCasCrop'),
}
with open('data/processed/feature_groups_monthly.json') as _f: _g = json.load(_f)
_bio, _econ, _hist = len(_g['biophysical']), len(_g['economic']), len(_g['historical'])
_N = 64
_batch = {
    'x_bio': torch.randn(_N, _bio, device=device),
    'x_econ': torch.randn(_N, _econ, device=device),
    'x_hist': torch.randn(_N, _hist, device=device),
    'price_shocks': torch.randn(_N, 1, device=device),
    'edge_index': torch.stack([torch.arange(_N), torch.arange(_N)]).to(device),
    'edge_attr': None,
}
for name, (mod, cls) in _reg.items():
    try:
        model = getattr(importlib.import_module(mod), cls)(
            bio_input_dim=_bio, econ_input_dim=_econ, hist_dim=_hist,
            latent_dim=64, hidden_dim=64, num_heads=4, dropout=0.3).to(device)
        out = model(_batch)
        loss = out['waste_logits'].sum(); loss.backward()
        print(f'  {name}: OK ({sum(p.numel() for p in model.parameters()):,} params)')
        del model, out, loss
    except Exception as e:
        print(f'  {name}: FAILED - {e}'); raise
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f'All models pass on {device}')

---
## Experiment 1: Main Temporal Ablation

In [ ]:
%%time
models = ['temporal_local', 'temporal_gat', 'temporal_symmetric', 'temporal_cascrop']
ss = ' '.join(str(s) for s in SEEDS)
t0 = time.time()

for i, m in enumerate(models):
    print(f'\n{"="*50}\n[{i+1}/{len(models)}] {m}\n{"="*50}')
    try:
        r = subprocess.run(
            f'python scripts/04_train_temporal.py --models {m} --seeds {ss} '
            f'--epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --resume',
            shell=True, capture_output=True, text=True, timeout=7200)
        for line in r.stdout.strip().split('\n')[-15:]: print(line)
        if r.returncode != 0:
            print(f'EXIT CODE: {r.returncode}')
            if r.stderr: print(f'STDERR:\n{r.stderr[-1000:]}')
    except subprocess.TimeoutExpired: print('Timeout')
    except Exception as e: print(f'Error: {e}')
    backup()
    eta = (time.time()-t0)/(i+1)*(len(models)-i-1)
    print(f'ETA: {eta/60:.0f} min')
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Results
if os.path.exists('results/temporal_results.json'):
    with open('results/temporal_results.json') as f: res = json.load(f)
    df = pd.DataFrame(res)
    print(f'{"Model":<25} {"AUC-ROC":>12} {"F1":>8}')
    print('-'*50)
    for m in models:
        d = df[(df['model']==m) & (df['test_auc_roc']>0)]
        if len(d):
            print(f'{m:<25} {d["test_auc_roc"].mean():.3f}+/-{d["test_auc_roc"].std():.3f}  {d["test_f1"].mean():.3f}')
    
    c = df[df['model']=='temporal_cascrop']['test_auc_roc'].mean()
    l = df[df['model']=='temporal_local']['test_auc_roc'].mean()
    g = df[df['model']=='temporal_gat']['test_auc_roc'].mean()
    s = df[df['model']=='temporal_symmetric']['test_auc_roc'].mean()
    print(f'\nH1 Graph>Local:    {c-l:+.4f}')
    print(f'H2 ECMP>GAT:       {c-g:+.4f}')
    print(f'H3 Asym>Sym:       {c-s:+.4f}')
else:
    print('No results yet')

---
## Experiment 2: Graph Perturbation

In [ ]:
%%time
import numpy as np, shutil
from pathlib import Path

# Build shuffled graph
g = np.load('data/graphs/combined_graph.npz'); np.random.seed(42)
np.savez('data/graphs/shuffled.npz',
         edge_index=np.array([g['edge_index'][0], np.random.permutation(g['edge_index'][1])]),
         edge_weight=g['edge_weight'])

# Back up main results and checkpoints before perturbation
shutil.copy('data/graphs/combined_graph.npz', 'data/graphs/combined_backup.npz')
for f in Path('checkpoints').glob('temporal_cascrop_*.pt'):
    shutil.copy(f, f'{f}.main_bak')
if os.path.exists('results/temporal_results.json'):
    shutil.copy('results/temporal_results.json', 'results/temporal_results_main.json')

# Swap in shuffled graph and train
shutil.copy('data/graphs/shuffled.npz', 'data/graphs/combined_graph.npz')
ss3 = ' '.join(str(s) for s in SEEDS[:3])
r = subprocess.run(
    f'python scripts/04_train_temporal.py --models temporal_cascrop --seeds {ss3} '
    f'--epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0',
    shell=True, capture_output=True, text=True, timeout=3600)
for line in r.stdout.strip().split('\n')[-10:]: print(line)
if r.returncode != 0 and r.stderr: print(f'STDERR: {r.stderr[-300:]}')

# Save perturbation results separately
if os.path.exists('results/temporal_results.json'):
    shutil.copy('results/temporal_results.json', 'results/perturbation_results.json')

# Restore originals
shutil.copy('data/graphs/combined_backup.npz', 'data/graphs/combined_graph.npz')
for f in Path('checkpoints').glob('temporal_cascrop_*.pt.main_bak'):
    shutil.copy(f, str(f).replace('.main_bak', ''))
    f.unlink()
if os.path.exists('results/temporal_results_main.json'):
    shutil.copy('results/temporal_results_main.json', 'results/temporal_results.json')

# Compare real vs shuffled
if os.path.exists('results/perturbation_results.json'):
    with open('results/perturbation_results.json') as f: pert = json.load(f)
    with open('results/temporal_results.json') as f: main = json.load(f)
    pdf = pd.DataFrame(pert)
    mdf = pd.DataFrame(main)
    pert_auc = pdf[pdf['model']=='temporal_cascrop']['test_auc_roc'].mean()
    main_auc = mdf[mdf['model']=='temporal_cascrop']['test_auc_roc'].mean()
    print(f'\nPerturbation Analysis:')
    print(f'  Real graph AUC:     {main_auc:.4f}')
    print(f'  Shuffled graph AUC: {pert_auc:.4f}')
    print(f'  Delta:              {main_auc - pert_auc:+.4f}')
    print(f'  Graph signal: {"CONFIRMED" if main_auc > pert_auc else "NOT CONFIRMED"}')

if torch.cuda.is_available(): torch.cuda.empty_cache()

---
## Results + Figure

In [ ]:
if os.path.exists('results/temporal_results.json'):
    with open('results/temporal_results.json') as f: res = json.load(f)
    df = pd.DataFrame(res)
    
    # Statistical tests
    from evaluation.statistical_tests import paired_ttest_across_seeds
    ca = sorted(df[df['model']=='temporal_cascrop']['test_auc_roc'].tolist())
    if len(ca) >= 2:
        print(f'{"Comparison":<40} {"DAUC":>7} {"p":>8} {"Sig":>5}')
        print('-'*63)
        for m in ['temporal_local','temporal_gat','temporal_symmetric']:
            ma = sorted(df[df['model']==m]['test_auc_roc'].tolist())
            if len(ma) != len(ca): continue
            t = paired_ttest_across_seeds(ca, ma)
            sig = '***' if t['p_value']<.001 else '**' if t['p_value']<.01 else '*' if t['p_value']<.05 else 'n.s.'
            print(f'TemporalCasCrop vs {m:<20} {t["mean_diff"]:>+.4f} {t["p_value"]:>8.4f} {sig:>5}')
    
    c = df[df['model']=='temporal_cascrop']['test_auc_roc'].mean()
    l = df[df['model']=='temporal_local']['test_auc_roc'].mean()
    g = df[df['model']=='temporal_gat']['test_auc_roc'].mean()
    s = df[df['model']=='temporal_symmetric']['test_auc_roc'].mean()
    print(f'\nH1 Graph>Local:    {c-l:+.4f} {"CONFIRMED" if c>l else "FAILED"}')
    print(f'H2 ECMP>GAT:       {c-g:+.4f} {"CONFIRMED" if c>g else "FAILED"}')
    print(f'H3 Asym>Sym:       {c-s:+.4f} {"CONFIRMED" if c>s else "FAILED"}')
else:
    print('No results yet')

In [ ]:
# Figure
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'font.size':9,'figure.dpi':300})

if 'df' in dir() and len(df) > 0:
    mo = ['temporal_local','temporal_gat','temporal_symmetric','temporal_cascrop']
    dn = ['T1: Local\n(GRU only)','T2: GAT\n(GRU+GAT)','T3: Symmetric\n(GRU+SymECMP)','T4: CasCrop\n(GRU+AsymECMP)']
    co = ['#7f8c8d','#e67e22','#9b59b6','#e74c3c']
    ms = [df[df['model']==m]['test_auc_roc'].mean() for m in mo]
    ss_ = [df[df['model']==m]['test_auc_roc'].std() if len(df[df['model']==m])>1 else 0 for m in mo]
    
    fig, ax = plt.subplots(figsize=(7,3.5))
    bars = ax.bar(range(4), ms, 0.6, yerr=ss_, capsize=4, color=co, edgecolor='k', linewidth=.5)
    for b, v in zip(bars, ms):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.008, f'{v:.3f}', ha='center', fontsize=7)
    ax.set_xticks(range(4)); ax.set_xticklabels(dn, fontsize=7)
    ax.set_ylim(0.7, 1.0); ax.set_ylabel('AUC-ROC'); ax.grid(axis='y', alpha=0.3)
    ax.set_title('Temporal Cascade Ablation (Monthly Data, 2022-2024)', fontweight='bold')
    plt.tight_layout()
    fig.savefig('paper/figures/fig3_temporal_ablation.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: paper/figures/fig3_temporal_ablation.pdf')

---
## Download

In [ ]:
backup()
!tar czf /content/cascrop_temporal_results.tar.gz results/ paper/figures/ paper/tables/ checkpoints/
try:
    from google.colab import files; files.download('/content/cascrop_temporal_results.tar.gz')
except: print('Download from Files or Drive')
print('DONE')